# Do Enem à graduação: minha parte de dados e gráficos

**Brasil · 2020 a 2024 · Pandas e Plotly**

Esta entrega ocupa somente `data/` e `notebooks/`, conforme a divisão de tarefas do grupo.
O notebook usa as classes de `src/`, as mesmas do painel, para leitura, exploração, indicadores e cinco gráficos.
As tabelas oficiais tratadas acompanham o notebook; os arquivos brutos ficam fora do Git.

**Perguntas:** como evoluíram as inscrições dos concluintes da rede pública no Enem?
Como se distribuem as redes registradas e os registros sem informação?
Como a série se apresenta ao lado do ingresso na graduação?

**Limite:** inscrições no Enem não identificam ingresso pelo Enem. As séries agregadas não acompanham as mesmas pessoas.
O roteiro e as limitações dos dados continuam explicados nas próximas seções.

## 1. Ambiente e arquivos

Abra este notebook na raiz do projeto ou em `notebooks/`. Ele procura `data/processed/` automaticamente.
Dependências: **pandas, plotly e openpyxl**; use um kernel Python no VS Code ou Jupyter.
Se precisar instalar, execute `%pip install pandas plotly openpyxl` em uma célula do seu ambiente.
Para repetir o download opcional dos originais, também é necessário `truststore`.

Não é preciso alterar o `requirements.txt` do grupo para entregar esta contribuição.
Os resultados e as quatro figuras já estão salvos no notebook.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/enem_brasil_rede.csv').exists()), None)
if ROOT is None:
    raise RuntimeError('Coloque data/ e notebooks/ na raiz do mesmo projeto.')
PROCESSED = ROOT / 'data/processed'
sys.path.insert(0, str(ROOT))  # permite importar src/
print('Pasta de dados encontrada: data/processed/')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

Pasta de dados encontrada: data/processed/


### Classes do projeto

O notebook importa as classes de `src/`, as mesmas usadas pelo painel Streamlit.
Assim, tabelas, indicadores e gráficos daqui são idênticos aos do aplicativo, e uma correção vale para os dois.

In [2]:
from src.analise import AnalisadorColeta
from src.dashboard import CODIGOS_TRAJETORIA
from src.fontes import FonteCSV, FonteExcel, carregar_dados
from src.indicadores import CatalogoIndicadores
from src.visualizacao import Visualizador

## 2. Catálogo das fontes e cobertura

Usamos 15 sinopses anuais (cinco edições de cada uma das três bases) e um arquivo de trajetória da coorte 2020.
O catálogo traz links, abas, datas operacionais e hashes dos arquivos efetivamente processados.
Os painéis do Inep ajudam a consultar os dados, mas esta entrega extrai e confere os números nas planilhas.

| Base | Tabela | O que mede | Uso |
|---|---|---|---|
| Enem | 3.10 | Inscritos que cursavam o último ano e concluiriam naquele ano, por dependência administrativa | Gráficos principais |
| Censo Escolar | 1.25 | Matrículas em todas as séries do ensino médio regular | Contexto, sem usar como concluintes |
| Censo Superior | 4.4 | Ingressantes na graduação, presencial e EaD, todas as formas | Comparação descritiva |
| Trajetória | Coorte 2020, referências 2020–2024 | Vínculos com cursos, permanência e eventos anuais | Contexto e exploração |

Não foram necessários microdados do Enem: as cinco sinopses possuem a tabela 3.10 exigida para os gráficos.
O Enem 2020 foi aplicado em 2021; os anos deste notebook seguem a **edição** do exame.

In [3]:
fontes = pd.DataFrame(json.loads((PROCESSED / 'fontes.json').read_text(encoding='utf-8')))
display(fontes[['id', 'ano', 'aba', 'pagina', 'url', 'data_arquivo_local', 'sha256_zip']])
assert len(fontes) == 16
assert fontes['id'].is_unique

,id,ano,aba,pagina,url,data_arquivo_local,sha256_zip
0,enem_2020,2020,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:13:20.949470-03:00,976f4590904d3c582146cd57870774255ce468cbf0ea4d...
1,superior_2020,2020,4.4,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:21:23.060462-03:00,6172df8ef2591a1467a73f9d67e3a7b0d027ff286da49f...
2,basica_2020,2020,Ensino Médio 1.25,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/dados_abertos/sin...,2026-11-19T20:20:34.741928-03:00,a252b0e5e8668503b54dbc3ef8c91f67d99bca042e872c...
3,enem_2021,2021,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:18:08.169716-03:00,5fa5d9efa3232cadde903c13fdac22fdb87ce11ec6a4c8...
4,superior_2021,2021,4.4,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:23:08.904004-03:00,0483c5677331eaa2724ceddc064ccfbe7c94b3f7c791b0...
5,basica_2021,2021,Ensino Médio 1.25,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/dados_abertos/sin...,2026-11-19T20:21:16.805187-03:00,66861b1e0adc7e09e503ca3683f960f644707a69be7c6a...
6,enem_2022,2022,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:18:43.963508-03:00,3f8c3b7d696bc1f6f5d0b421f3b1a18e7f911c844a21ba...
7,superior_2022,2022,4.4,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:25:08.639771-03:00,6ce329f291618d0a68d2bb8e48a49cdeb0d383b5691e12...
8,basica_2022,2022,Ensino Médio 1.25,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/dados_abertos/sin...,2026-11-19T20:23:01.109022-03:00,811d291a9ad716906132e162e871c8e0f8fc09b092205a...
9,enem_2023,2023,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:18:45.653694-03:00,e6b7bfa46694966c23bb2ac5e5b8eb504065d8b99c9b53...


## 3. Conferência de uma amostra original

As sinopses têm títulos e cabeçalhos em várias linhas. Não usamos `skiprows=8` indiscriminadamente.
O extrator verifica o título, o ano e os campos, encontra a linha **Brasil** e deixa os subtotais de regiões e UFs de fora.
As células da linha nacional foram preservadas em `evidencias_extracao.json` para conferência mesmo sem os arquivos grandes.

Na tabela do Enem, **Pública = Federal + Estadual + Municipal**. A categoria sem informação não é atribuída a nenhuma rede.

In [4]:
evidencias = json.loads((PROCESSED / 'evidencias_extracao.json').read_text(encoding='utf-8'))
exemplo = next(e for e in evidencias if e['ano'] == 2020 and e['aba'] == '3.10')
print(exemplo['titulo'])
print('Linha nacional no Excel:', exemplo['linha_excel'])
display(pd.DataFrame(exemplo['celulas'].items(), columns=['celula', 'valor_original']))

# Se os brutos estiverem disponíveis, mostramos também a leitura direta da fonte.
fonte_2020 = fontes.loc[fontes.id.eq('enem_2020')].iloc[0]
caminho_bruto = ROOT / fonte_2020.arquivo_xlsx
if caminho_bruto.exists():
    amostra_original = FonteExcel(caminho_bruto, '3.10', limite_linhas=13).carregar()
    display(amostra_original.iloc[:13, :8])
else:
    print('Arquivo bruto não incluído no pacote. As células oficiais estão registradas acima.')

3.10 - Número de Concluintes no Enem segundo a Dependência Administrativa por Região Geográfica e Unidade da Federação - 2020
Linha nacional no Excel: 10


,celula,valor_original
0,A10,Brasil
1,C10,1395827
2,D10,46594
3,E10,630939
4,F10,8331
5,G10,218705
6,H10,491258


Arquivo bruto não incluído no pacote. As células oficiais estão registradas acima.


## 4. Leitura das tabelas tratadas com Pandas

Cada base permanece separada. O vínculo entre séries para a dispersão é somente o ano, sem associação individual.
`rede_origem` representa a escola de origem; `categoria_ies` representa a instituição superior e está como “Todas” nesta seleção.

In [5]:
enem = FonteCSV(PROCESSED / 'enem_brasil_rede.csv').carregar()
superior = FonteCSV(PROCESSED / 'superior_brasil.csv').carregar()
basica = FonteCSV(PROCESSED / 'basica_matriculas_brasil.csv').carregar()
trajetoria = FonteCSV(PROCESSED / 'trajetoria_coorte_2020.csv').carregar()
amostra_trajetoria = FonteCSV(PROCESSED / 'trajetoria_amostra_25_linhas.csv').carregar()

for nome, tabela in [('Enem', enem), ('Superior', superior), ('Básica', basica), ('Trajetória', trajetoria)]:
    print(nome, '→', tabela.shape[0], 'linhas e', tabela.shape[1], 'colunas')
    display(tabela.head())

Enem → 15 linhas e 7 colunas


,ano,abrangencia,rede_origem,inscritos_concluintes,total_concluintes_inscritos,fonte_id,aba
0,2020,Brasil,Pública,685864,1395827,enem_2020,3.10
1,2020,Brasil,Privada,218705,1395827,enem_2020,3.10
2,2020,Brasil,Não dispõe da informação,491258,1395827,enem_2020,3.10
3,2021,Brasil,Pública,603041,1150857,enem_2021,3.10
4,2021,Brasil,Privada,210765,1150857,enem_2021,3.10


Superior → 5 linhas e 9 colunas


,ano,abrangencia,categoria_ies,modalidade,forma_ingresso,ingressantes_graduacao,ingressantes_universidades,fonte_id,aba
0,2020,Brasil,Todas,Presencial e a distância,Todas,3765475,2032571,superior_2020,4.40
1,2021,Brasil,Todas,Presencial e a distância,Todas,3944897,2138197,superior_2021,4.40
2,2022,Brasil,Todas,Presencial e a distância,Todas,4756728,2584923,superior_2022,4.40
3,2023,Brasil,Todas,Presencial e a distância,Todas,4993992,2624163,superior_2023,4.40
4,2024,Brasil,Todas,Presencial e a distância,Todas,5010433,2675687,superior_2024,4.40


Básica → 10 linhas e 7 colunas


,ano,abrangencia,rede_origem,matriculas_em_regular,universo,fonte_id,aba
0,2020,Brasil,Pública,6624804,Todas as séries do ensino médio regular; não s...,basica_2020,Ensino Médio 1.25
1,2020,Brasil,Privada,925949,Todas as séries do ensino médio regular; não s...,basica_2020,Ensino Médio 1.25
2,2021,Brasil,Pública,6835399,Todas as séries do ensino médio regular; não s...,basica_2021,Ensino Médio 1.25
3,2021,Brasil,Privada,935158,Todas as séries do ensino médio regular; não s...,basica_2021,Ensino Médio 1.25
4,2022,Brasil,Pública,6895219,Todas as séries do ensino médio regular; não s...,basica_2022,Ensino Médio 1.25


Trajetória → 5 linhas e 15 colunas


,ano_referencia,vinculos_ingressantes_coorte,vinculos_permanencia,conclusoes_no_ano,desistencias_no_ano,falecimentos_no_ano,ausentes_qt_ingressante,ausentes_qt_permanencia,ausentes_qt_concluinte,ausentes_qt_desistencia,ausentes_qt_falecido,cursos_observados,ano_ingresso,fonte_id,unidade
0,2020,3859448,3214243,42358,602677,170,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
1,2021,3859448,2284306,151399,778112,426,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
2,2022,3859448,1554937,206928,522238,203,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
3,2023,3859448,1010329,243853,300419,336,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
4,2024,3859448,522934,296288,191002,105,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas


## 5. Tipos, ausências, duplicidades e estatísticas

Mostramos as verificações pedidas na Etapa 2 do enunciado: `info()`, `isna().sum()`, `duplicated()` e `describe()`.
As estatísticas descrevem contagens anuais agregadas, não notas nem características individuais de estudantes.
“Não dispõe da informação” é uma categoria explícita; por isso não aparece como `NaN` em `rede_origem`.

In [6]:
bases = {'Enem': (enem, ['ano', 'rede_origem']),
         'Educação Superior': (superior, ['ano']),
         'Educação Básica': (basica, ['ano', 'rede_origem']),
         'Trajetória': (trajetoria, ['ano_ingresso', 'ano_referencia'])}
for nome, (tabela, chave) in bases.items():
    print()
    print('BASE:', nome)
    tabela.info()
    display(tabela.isna().sum().rename('valores_ausentes').to_frame())
    print('Linhas repetidas:', tabela.duplicated().sum())
    print('Chaves repetidas:', tabela.duplicated(chave).sum())
    display(tabela.select_dtypes('number').describe())

print('Amostra de trajetória: dados por curso, coorte e ano de referência')
amostra_trajetoria.info()
display(amostra_trajetoria.isna().sum().to_frame('valores_ausentes'))
display(amostra_trajetoria.head())
perfil_trajetoria = json.loads((PROCESSED / 'perfil_trajetoria.json').read_text(encoding='utf-8'))
print('Diagnóstico do arquivo completo de trajetória:', perfil_trajetoria['registros'], 'registros')
display(pd.DataFrame({'tipo': perfil_trajetoria['tipos'],
                      'ausencias_base_completa': perfil_trajetoria['ausencias']}))
print(perfil_trajetoria['nota'])


BASE: Enem
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ano                          15 non-null     int64  
 1   abrangencia                  15 non-null     str    
 2   rede_origem                  15 non-null     str    
 3   inscritos_concluintes        15 non-null     int64  
 4   total_concluintes_inscritos  15 non-null     int64  
 5   fonte_id                     15 non-null     str    
 6   aba                          15 non-null     float64
dtypes: float64(1), int64(3), str(3)
memory usage: 1.4 KB


,valores_ausentes
ano,0
abrangencia,0
rede_origem,0
inscritos_concluintes,0
total_concluintes_inscritos,0
fonte_id,0
aba,0


Linhas repetidas: 0
Chaves repetidas: 0


,ano,inscritos_concluintes,total_concluintes_inscritos,aba
count,15.00,15.00,15.00,15.00
mean,"2,022.00","458,867.33","1,376,602.00",3.10
std,1.46,"256,731.10","155,953.74",0.00
min,"2,020.00",519.00,"1,150,857.00",3.10
25%,"2,021.00","228,524.50","1,317,560.00",3.10
50%,"2,022.00","442,658.00","1,395,827.00",3.10
75%,"2,023.00","705,193.50","1,401,164.00",3.10
max,"2,024.00","841,213.00","1,617,602.00",3.10



BASE: Educação Superior
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ano                         5 non-null      int64  
 1   abrangencia                 5 non-null      str    
 2   categoria_ies               5 non-null      str    
 3   modalidade                  5 non-null      str    
 4   forma_ingresso              5 non-null      str    
 5   ingressantes_graduacao      5 non-null      int64  
 6   ingressantes_universidades  5 non-null      int64  
 7   fonte_id                    5 non-null      str    
 8   aba                         5 non-null      float64
dtypes: float64(1), int64(3), str(5)
memory usage: 762.0 bytes


,valores_ausentes
ano,0
abrangencia,0
categoria_ies,0
modalidade,0
forma_ingresso,0
ingressantes_graduacao,0
ingressantes_universidades,0
fonte_id,0
aba,0


... (12 outputs left out)

,CO_IES,NO_IES,TP_CATEGORIA_ADMINISTRATIVA,TP_ORGANIZACAO_ACADEMICA,CO_CURSO,NO_CURSO,CO_REGIAO,CO_UF,CO_MUNICIPIO,TP_GRAU_ACADEMICO,TP_MODALIDADE_ENSINO,CO_CINE_ROTULO,NO_CINE_ROTULO,CO_CINE_AREA_GERAL,NO_CINE_AREA_GERAL,NU_ANO_INGRESSO,NU_ANO_REFERENCIA,NU_PRAZO_INTEGRALIZACAO,NU_ANO_INTEGRALIZACAO,NU_PRAZO_ACOMPANHAMENTO,NU_ANO_MAXIMO_ACOMPANHAMENTO,QT_INGRESSANTE,QT_PERMANENCIA,QT_CONCLUINTE,QT_DESISTENCIA,QT_FALECIDO,TAP,TCA,TDA,TCAN,TADA
0,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2020,6,2025,10,2029,100,100,0,0,0,100.00,0.00,0.00,0.00,0.00
1,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2021,6,2025,10,2029,100,97,0,3,0,97.00,0.00,3.00,0.00,3.00
2,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2022,6,2025,10,2029,100,92,1,4,0,92.00,1.00,7.00,1.00,4.00
3,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2023,6,2025,10,2029,100,91,1,0,0,91.00,2.00,7.00,1.00,0.00
4,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2024,6,2025,10,2029,100,34,57,0,0,34.00,59.00,7.00,57.00,0.00


Diagnóstico do arquivo completo de trajetória: 174240 registros


,tipo,ausencias_base_completa
CO_IES,object,0
NO_IES,str,0
TP_CATEGORIA_ADMINISTRATIVA,object,0
TP_ORGANIZACAO_ACADEMICA,object,0
CO_CURSO,object,0
NO_CURSO,str,0
CO_REGIAO,object,26230
CO_UF,object,26230
CO_MUNICIPIO,object,26230
TP_GRAU_ACADEMICO,object,0


Ausências geográficas preservadas; agregação nacional inclui esses cursos.


## 6. Tratamento e reconciliação

- Preservamos os arquivos brutos. Apenas os CSVs agregados entram no Git.
- Selecionamos a linha Brasil, sem somar Brasil + regiões + estados.
- Conferimos a soma das dependências do Enem, das organizações acadêmicas no Censo Superior e das matrículas por série/dependência no Censo Escolar.
- Não excluímos linhas nem removemos duplicidades sem diagnóstico. Uma chave duplicada interrompe a preparação.
- Não transformamos vazios ou símbolos em zero. Ausências numéricas necessárias à reconciliação interrompem o processamento.
- Os dados de trajetória resumem a mesma coorte em cada referência; não somamos os ingressantes de 2020 novamente a cada ano.

In [7]:
conferencia = enem.groupby('ano').agg(
    soma_categorias=('inscritos_concluintes', 'sum'),
    total_publicado=('total_concluintes_inscritos', 'first'))
conferencia['confere'] = conferencia.soma_categorias.eq(conferencia.total_publicado)
display(conferencia)
assert conferencia.confere.all()
assert set(enem.ano) == set(range(2020, 2025))
assert not enem.duplicated(['ano', 'rede_origem']).any()
assert (superior.ingressantes_universidades <= superior.ingressantes_graduacao).all()
assert len(evidencias) == 15
print('Totais nacionais e chaves conferidos.')

,soma_categorias,total_publicado,confere
ano,,,
2020,1395827,1395827,True
2021,1150857,1150857,True
2022,1317560,1317560,True
2023,1401164,1401164,True
2024,1617602,1617602,True


Totais nacionais e chaves conferidos.


## 7. Indicadores exploratórios

Calculamos contagens de inscrições por rede, o total, a participação da rede pública **entre todos os concluintes inscritos**, a proporção sem informação e a variação anual da contagem pública.

**Fórmulas:** participação pública = inscrições públicas / total de concluintes inscritos × 100;
sem informação = inscrições sem informação / mesmo total × 100;
variação anual = (contagem pública atual / anterior − 1) × 100.

São indicadores descritivos. Não representam a proporção de todos os concluintes do Brasil que fizeram o Enem.
Não calculamos a taxa de transição para a graduação, pois ela exigiria identificar a coorte de egressos e seus ingressos no ano seguinte.

In [8]:
resumo = AnalisadorColeta(enem).resumo()
display(resumo)
print('O primeiro ano não tem variação anual calculável nesta janela:',
      pd.isna(resumo.loc[0, 'variacao_anual_publica_pct']))
ano_2024 = resumo.loc[resumo.ano.eq(2024)].iloc[0]
print(f'2024: privada = {ano_2024.inscritos_privada:,.0f}; '
      f'sem informação = {ano_2024.inscritos_sem_informacao:,.0f} '
      f'({ano_2024.sem_informacao_pct:.2f}% do total).')
print('A causa dessa mudança acentuada da categoria privada não foi confirmada.')

print('Estatísticas descritivas das inscrições por rede, 2020–2024:')
display(AnalisadorColeta(enem).estatisticas())

catalogo = CatalogoIndicadores.padrao()
dados = carregar_dados()
print(f'Os {len(catalogo)} indicadores do painel:')
display(catalogo.tabela(dados))

,ano,inscritos_sem_informacao,inscritos_privada,inscritos_publica,total_concluintes_inscritos,participacao_publica_no_total_pct,sem_informacao_pct,variacao_anual_publica_pct
0,2020,491258,218705,685864,1395827,49.14,35.19,NaN
1,2021,337051,210765,603041,1150857,52.40,29.29,-12.08
2,2022,365616,227421,724523,1317560,54.99,27.75,20.14
3,2023,442658,229628,728878,1401164,52.02,31.59,0.60
4,2024,775870,519,841213,1617602,52.00,47.96,15.41


O primeiro ano não tem variação anual calculável nesta janela: True
2024: privada = 519; sem informação = 775,870 (47.96% do total).
A causa dessa mudança acentuada da categoria privada não foi confirmada.
Estatísticas descritivas das inscrições por rede, 2020–2024:


,rede_origem,edicoes,media,desvio_padrao,minimo,quartil_1,mediana,quartil_3,maximo,coef_variacao_pct
0,Pública,5.00,"716,703.80","85,997.41","603,041.00","685,864.00","724,523.00","728,878.00","841,213.00",12.00
1,Privada,5.00,"177,407.60","99,166.63",519.00,"210,765.00","218,705.00","227,421.00","229,628.00",55.90
2,Não dispõe da informação,5.00,"482,490.60","175,033.80","337,051.00","365,616.00","442,658.00","491,258.00","775,870.00",36.28


Os 8 indicadores do painel:


,ano,inscritos_publica,participacao_publica_no_total_pct,sem_informacao_pct,variacao_anual_publica_pct,ingressantes_graduacao,permanencia_coorte,conclusao_acumulada_coorte,desistencia_acumulada_coorte
0,2020,"685,864.00",49.14,35.19,NaN,"3,765,475.00",83.29,1.10,15.62
1,2021,"603,041.00",52.40,29.29,-12.08,"3,944,897.00",59.20,5.02,35.78
2,2022,"724,523.00",54.99,27.75,20.14,"4,756,728.00",40.30,10.38,49.32
3,2023,"728,878.00",52.02,31.59,0.60,"4,993,992.00",26.19,16.71,57.11
4,2024,"841,213.00",52.00,47.96,15.41,"5,010,433.00",13.55,24.39,62.06


## 8. Cinco gráficos interativos

As figuras são produzidas pela classe `Visualizador`, que recebe tabelas prontas e devolve figuras Plotly.
Podem ser reutilizadas com `st.plotly_chart(figura)` pelo colega responsável pelo dashboard.

In [9]:
visualizador = Visualizador()
dados['trajetoria_pct'] = catalogo.tabela_longa(dados, CODIGOS_TRAJETORIA)
figuras = visualizador.todas(dados)
# O formato Plotly é mantido na saída para ambientes compatíveis, como Jupyter.
# No GitHub, use também os HTMLs, pois a renderização do notebook pode ser estática.
for nome, figura in figuras.items():
    print(nome)
    display(figura)

01_linhas


02_barras


03_setores


04_dispersao


05_trajetoria


### Como interpretar

**Linhas:** a contagem pública cai em 2021 e cresce nas edições seguintes. A leitura exige considerar a cobertura da rede e as particularidades da pandemia; o gráfico sozinho não explica as causas.

**Barras:** comparam números absolutos das redes registradas. A queda acentuada da categoria privada em 2024 é um alerta de comparabilidade, não evidência suficiente de uma queda real da participação.

**Setores:** os registros sem informação da rede integram o total de 2024. Não recalculamos a composição omitindo essa categoria.

**Dispersão:** cada ponto representa um ano, comparando inscrições de concluintes públicos no Enem com ingressos de todas as origens e formas na graduação. Não há ligação entre pessoas, teste de correlação ou conclusão de causa e efeito.

**Trajetória:** acompanha os vínculos dos ingressantes de 2020 no curso de ingresso. A permanência cai a cada ano, enquanto conclusão e desistência acumulam; as três curvas somam 100%. Desistir do curso não equivale a deixar o ensino superior.

## 9. Exportação dos gráficos

Os gráficos publicados no site do projeto ficam em `docs/graficos/`. Para regravá-los, mude `EXPORTAR_GRAFICOS` para `True`.
Por padrão nada é gravado, para que executar o notebook não altere arquivos versionados.
O resumo `data/processed/indicadores_exploratorios.csv` é gerado por `scripts/preparar_dados.py`.

In [10]:
EXPORTAR_GRAFICOS = False  # True regrava os HTML de docs/graficos/

if EXPORTAR_GRAFICOS:
    pasta = visualizador.exportar(figuras, ROOT / 'docs/graficos')
    print('Gráficos exportados:', sorted(p.name for p in pasta.glob('*.html')))
else:
    print('Exportação desativada; os gráficos publicados estão em docs/graficos/.')

Exportação desativada; os gráficos publicados estão em docs/graficos/.


## 10. Minha entrega ao grupo

- `data/processed/`: tabelas nacionais, catálogo de fontes, evidências, diagnóstico e gráficos exportados.
- `data/raw/`: arquivos originais do Inep apenas no computador; o `.gitignore` dessa própria pasta impede o envio dos brutos.
- `notebooks/01_exploracao.ipynb`: exploração executada com as classes de `src/` e cinco gráficos interativos.

O notebook reutiliza as classes de `src/` e a preparação de `scripts/preparar_dados.py`, sem cópias do código.

As conclusões permitidas e as limitações são as mesmas: não se mede aprovação nem ingresso individual pelo Enem,
e a categoria privada de 2024 exige cautela devido à alteração acentuada dos registros na fonte.

As fontes oficiais, links, abas e hashes estão em `data/processed/fontes.json`.
Os caminhos `*_na_coleta_original` documentam a organização anterior da coleta;
`arquivo_zip` e `arquivo_xlsx` indicam a localização esperada nesta entrega.

## Apêndice opcional: repetir a coleta e o tratamento

As tabelas já foram preparadas e validadas. **Não é necessário rodar o download para usar esta entrega.**
A célula abaixo executa `scripts/preparar_dados.py`, o mesmo código do projeto. Ele lê os ZIPs de `data/raw/`,
confere cabeçalhos e totais num diretório temporário e só então substitui os arquivos de `data/processed/`.
Os ZIPs oficiais somam alguns GB. Depois, execute novamente as células de leitura, análise e gráficos.

In [11]:
REPROCESSAR_FONTES = False  # True: reprocessa os ZIPs de data/raw/
BAIXAR_FONTES = False       # True: baixa do Inep os ZIPs ausentes antes de reprocessar

if REPROCESSAR_FONTES:
    import subprocess
    opcao = '--baixar' if BAIXAR_FONTES else '--brutos'
    subprocess.run([sys.executable, str(ROOT / 'scripts/preparar_dados.py'), opcao], check=True)
    print('Preparação concluída. Reexecute as células de leitura, análise e gráficos.')
else:
    print('Coleta opcional desativada. Usamos os CSVs oficiais tratados que acompanham esta entrega.')

Coleta opcional desativada. Usamos os CSVs oficiais tratados que acompanham esta entrega.
